In [1]:
import torch
import sys
print("Interpreter:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Interpreter: d:\maga25\NIR\.venv\Scripts\python.exe
CUDA available: True
GPU name: NVIDIA GeForce GTX 1660


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")
if device.type == 'cuda':
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"Всего видеопамяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Используемое устройство: cuda
Видеокарта: NVIDIA GeForce GTX 1660
Всего видеопамяти: 6.44 GB


In [3]:
# Standard
import random

import numpy as np
import pandas as pd
import torch

# Third Party
from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments,
)

# First Party
from tsfm_public.toolkit.dataset import ForecastDFDataset
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.util import select_by_index

d:\maga25\NIR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

------------------------------------------------------------------------


Дообучение на датасете weather

In [5]:
# Путь к вашему файлу
file_path = './weather.csv' 

# Загрузка данных
df = pd.read_csv(file_path)
print(df.shape)
print(df.info())


(52696, 22)
<class 'pandas.DataFrame'>
RangeIndex: 52696 entries, 0 to 52695
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  52696 non-null  str    
 1   p (mbar)              52696 non-null  float64
 2   T (degC)              52696 non-null  float64
 3   Tpot (K)              52696 non-null  float64
 4   Tdew (degC)           52696 non-null  float64
 5   rh (%)                52696 non-null  float64
 6   VPmax (mbar)          52696 non-null  float64
 7   VPact (mbar)          52696 non-null  float64
 8   VPdef (mbar)          52696 non-null  float64
 9   sh (g/kg)             52696 non-null  float64
 10  H2OC (mmol/mol)       52696 non-null  float64
 11  rho (g/m**3)          52696 non-null  float64
 12  wv (m/s)              52696 non-null  float64
 13  max. wv (m/s)         52696 non-null  float64
 14  wd (deg)              52696 non-null  float64
 15  rain (mm)         

In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import pandas as pd
import numpy as np
import torch
import random

torch.set_default_device('cpu')

from transformers import (
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.dataset import ForecastDFDataset

# --- Настройки ---
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

context_length = 96
forecast_horizon = 96
patch_length = 12
batch_size = 4
num_epochs = 5
dataloader_workers = 0
pin_memory = False

# Уменьшаем объём обучающих данных для ускорения (используем первые 20%)
train_fraction = 0.2

# --- Загрузка данных ---
df = pd.read_csv('weather.csv', parse_dates=['date'])
forecast_columns = [col for col in df.columns if col != 'date']

n = len(df)
train_end = int(n * train_fraction)
valid_end = int(n * (train_fraction + 0.2))
if valid_end <= train_end:
    valid_end = int(n * 0.4)

train_data = df.iloc[:train_end]
valid_data = df.iloc[train_end:valid_end]
test_data = df.iloc[valid_end:]

print(f"Train: {len(train_data)}, Valid: {len(valid_data)}, Test: {len(test_data)}")

# --- Препроцессинг ---
tsp_weather = TimeSeriesPreprocessor(
    timestamp_column='date',
    id_columns=[],
    target_columns=forecast_columns,
    scaling=True,
)
tsp_weather.train(train_data)

train_dataset = ForecastDFDataset(
    tsp_weather.preprocess(train_data),
    id_columns=[],
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)
valid_dataset = ForecastDFDataset(
    tsp_weather.preprocess(valid_data),
    id_columns=[],
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)
test_dataset = ForecastDFDataset(
    tsp_weather.preprocess(test_data),
    id_columns=[],
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)

print(f"Train samples: {len(train_dataset)}, Valid: {len(valid_dataset)}, Test: {len(test_dataset)}")
if len(valid_dataset) == 0:
    raise ValueError("Валидационный датасет пуст.")

# --- Адаптация модели ---
model_path = "./patchtst_etth1_model"
old_config = PatchTSTConfig.from_pretrained(model_path)
new_config = PatchTSTConfig(
    num_input_channels=len(forecast_columns),
    context_length=context_length,
    patch_length=patch_length,
    prediction_length=forecast_horizon,
    d_model=128,
    num_attention_heads=old_config.num_attention_heads,
    num_hidden_layers=old_config.num_hidden_layers,
    ffn_dim=old_config.ffn_dim,
    dropout=old_config.dropout,
    head_dropout=old_config.head_dropout,
    pooling_type=old_config.pooling_type,
    channel_attention=old_config.channel_attention,
    scaling=old_config.scaling,
    loss=old_config.loss,
    pre_norm=old_config.pre_norm,
    norm_type=old_config.norm_type,
)
model = PatchTSTForPrediction.from_pretrained(
    model_path,
    config=new_config,
    ignore_mismatched_sizes=True
)
model = model.to('cpu')
print("✅ Модель адаптирована для Weather (21 канал)")

# --- Аргументы обучения (без compute_metrics) ---
train_args = TrainingArguments(
    output_dir="./checkpoint/patchtst/finetune/weather/",
    learning_rate=1e-4,
    num_train_epochs=num_epochs,
    do_eval=True,
    eval_strategy="epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    dataloader_num_workers=dataloader_workers,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   # используем встроенную потерю
    greater_is_better=False,
    weight_decay=0.01,
    label_names=["future_values"],
    fp16=False,
    dataloader_pin_memory=pin_memory,
    logging_steps=10,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    # compute_metrics удалён
    callbacks=[early_stopping],
)

print("\n🚀 Начинаем дообучение на Weather (только CPU, уменьшенные данные)...")
trainer.train()

print("\n📊 Оценка на тесте:")
test_metrics = trainer.evaluate(test_dataset)
print(f"Test loss (MSE в нормированном пространстве): {test_metrics['eval_loss']:.4f}")

# Сохранение
model.save_pretrained("./patchtst_weather_finetuned_cpu_final")
tsp_weather.save_pretrained("./patchtst_weather_preprocessor_cpu_final")
print("✅ Модель и препроцессор сохранены.")

Train: 10539, Valid: 10539, Test: 31618
Train samples: 10348, Valid: 10348, Test: 31427


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 4169.23it/s]
[transformers] PatchTSTForPrediction LOAD REPORT from: ./patchtst_etth1_model
Key                                           | Status   |                                                                                            
----------------------------------------------+----------+--------------------------------------------------------------------------------------------
head.projection.weight                        | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([96, 5376]) vs model:torch.Size([96, 10880])
model.encoder.positional_encoder.position_enc | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([42, 128]) vs model:torch.Size([85, 128])   

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


✅ Модель адаптирована для Weather (21 канал)

🚀 Начинаем дообучение на Weather (только CPU, уменьшенные данные)...


Epoch,Training Loss,Validation Loss
1,0.793133,1.869251
2,0.569626,1.801245
3,0.478154,1.847141
4,0.709122,1.805571
5,0.593157,1.829589


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 39.73it/s]



📊 Оценка на тесте:


Training Loss,Validation Loss,Epoch
0.593157,1.978968,5


Test loss (MSE в нормированном пространстве): 1.9790


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 47.58it/s]
INFO:p-27900:t-28328:processor.py:save_pretrained:Feature extractor saved in ./patchtst_weather_preprocessor_cpu_final\preprocessor_config.json


✅ Модель и препроцессор сохранены.
